# 02 — MOFA tools over MCP (FastMCP + langchain-mcp-adapters)

Same analysis tools as [`01_tools_v1`](01_tools_v1.ipynb), but now served by a
**FastMCP** server (`server/mofa_mcp_server.py`) and pulled into the agent via
**langchain-mcp-adapters**. The agent code is almost identical to notebook 01;
only the *source* of the tools changes.

```text
LLM client  ->  MCP server (FastMCP)  ->  cached MOFA model  ->  structured result  ->  LLM answer
```

**Task (flagship):** Which MOFA factor is most associated with breast-cancer
subtype, and what drives it?

### Model backend — bring-your-own-key

Reads `ANTHROPIC_API_KEY` from `.env`; run with the **`eccb`** kernel. Only the
agent-run cell in Section 6 spends API tokens — every cell above it is local
(the server subprocess loads the cached model; no LLM involved).

### Why MCP here?

For this practical the MCP server buys little over the local functions in
notebook 01 — and that's the point: the *task* is identical, so the only thing
that changes is the **interface**. MCP starts to pay off when the tools front
something you don't want inside every notebook process — here, a **1.1 GB**
`omics.pkl` and a fitted model. The server loads that **once** at startup and
exposes only small, audited, read-only operations; many clients can reuse it.

## 0. Environment & API key

In [1]:
from pathlib import Path
import os, sys, json

from dotenv import load_dotenv

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))
load_dotenv(PROJECT_ROOT / ".env")

if not os.environ.get("ANTHROPIC_API_KEY"):
    raise RuntimeError("ANTHROPIC_API_KEY not found. Add it to a .env in the project root.")
print("ANTHROPIC_API_KEY loaded:", bool(os.environ.get("ANTHROPIC_API_KEY")))

SERVER_PATH = PROJECT_ROOT / "server" / "mofa_mcp_server.py"
print("MCP server:", SERVER_PATH.name)

ANTHROPIC_API_KEY loaded: True
MCP server: mofa_mcp_server.py


## 1. The MCP server

`server/mofa_mcp_server.py` uses **FastMCP** to expose three MCP primitives, all
backed by the cached MOFA model:

- **Tools** (model-callable, read-only): `data_summary`, `split_summary`,
  `active_factors`, `factor_view_r2`, `factor_subtype_association`,
  `top_features_for_factor`, `classify_subtype_from_factors`,
  `train_vs_test_subtype_association`
- **Resource**: `mofa://summary` — a compact model/cohort summary to read into context
- **Prompt**: `interpret_factor(factor)` — a reusable interpretation template

The server loads the aligned omics + fitted model **once at startup**; `fit_mofa`
is not exposed. The clients below launch it as a stdio subprocess.

## 2. Under the hood: talk to MCP directly (no adapter)

Before the LangChain adapter, connect with the MCP SDK's own client to see what
MCP *is*: open a `ClientSession` over stdio, handshake, enumerate every
primitive, then call a tool over the wire. (No LLM — this is free.)

In [2]:
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

server_params = StdioServerParameters(command=sys.executable, args=[str(SERVER_PATH)])

async with stdio_client(server_params) as (read, write):
    async with ClientSession(read, write) as session:
        init = await session.initialize()
        print(f"Connected to: {init.serverInfo.name}")
        caps = init.capabilities
        print("Server advertises:",
              "tools" if caps.tools else "", "resources" if caps.resources else "",
              "prompts" if caps.prompts else "")

        print("\n--- TOOLS (the model may choose to call these) ---")
        for t in (await session.list_tools()).tools:
            args = ", ".join(t.inputSchema.get("properties", {}))
            ro = getattr(t.annotations, "readOnlyHint", None)
            print(f"  {t.name}({args})   readOnlyHint={ro}")

        print("\n--- RESOURCES (data to read into context) ---")
        for r in (await session.list_resources()).resources:
            print(f"  {r.uri}  ({r.name})")
        summary = await session.read_resource("mofa://summary")
        print("  read mofa://summary ->", summary.contents[0].text[:80].replace(chr(10), " "), "...")

        print("\n--- PROMPTS (server-provided templates) ---")
        for p in (await session.list_prompts()).prompts:
            args = ", ".join(a.name for a in (p.arguments or []))
            print(f"  {p.name}({args})")

        print("\n--- tools/call factor_subtype_association ---")
        called = await session.call_tool("factor_subtype_association", {})
        print("  ", called.content[0].text[:120].replace(chr(10), " "))

Connected to: eccb2026-mofa
Server advertises: tools resources prompts

--- TOOLS (the model may choose to call these) ---
  data_summary()   readOnlyHint=True
  split_summary()   readOnlyHint=True
  active_factors()   readOnlyHint=True
  factor_view_r2(factor)   readOnlyHint=True
  factor_subtype_association()   readOnlyHint=True
  top_features_for_factor(factor, view, n)   readOnlyHint=True
  classify_subtype_from_factors()   readOnlyHint=True
  train_vs_test_subtype_association()   readOnlyHint=True

--- RESOURCES (data to read into context) ---
  mofa://summary  (mofa_summary)
  read mofa://summary -> {   "n_patients": 603,   "views": [     "transcriptomics",     "proteomics",     ...

--- PROMPTS (server-provided templates) ---
  interpret_factor(factor)

--- tools/call factor_subtype_association ---
   {   "factor": "Factor2",   "eta_squared": 0.743 }


The block above never imported LangChain — that is MCP directly:

- **JSON-RPC 2.0 over a transport.** Here the transport is `stdio`; it could be
  HTTP. On that wire MCP defines methods — `initialize`, `tools/list`,
  `tools/call`, `resources/list`, `resources/read`, `prompts/list`,
  `prompts/get`. Layering: **stdio/HTTP → JSON-RPC → MCP semantics**.
- **Tools are one primitive of several:**

| Primitive | Driven by | What it is | In our server |
|-----------|-----------|------------|---------------|
| **Tools** | the model | functions the model may *choose* to call | the 8 analysis functions |
| **Resources** | the app/user | data to *read into context* | `mofa://summary` |
| **Prompts** | the user | reusable, parameterized templates | `interpret_factor` |
| **Sampling** | the server | server asks the *client's* LLM to generate | (not used) |
| **Elicitation** | the server | server asks the *human* mid-call | (not used) |
| **Roots** | the client | filesystem/scope boundaries | (not used) |

- **LLM-facing metadata:** each tool carried `readOnlyHint=True`. A client can
  use that to auto-run vs. pause for human approval — meaningful precisely
  because a non-deterministic model is the caller. Generic RPC has no such notion.

## 3. Framework adaptation: `langchain-mcp-adapters`

We call **`get_tools()`**, so only the *tools* primitive flows into the agent.
The same client also exposes `get_resources()` / `get_prompt()`. Nothing about
MCP is thrown away — the adapter **is** an MCP client speaking the identical wire
protocol as section 2; `get_tools()` just converts the one primitive a
tool-calling loop consumes.

In [3]:
from langchain_mcp_adapters.client import MultiServerMCPClient

mcp_client = MultiServerMCPClient({
    "mofa": {"command": sys.executable, "args": [str(SERVER_PATH)], "transport": "stdio"}
})

tools = await mcp_client.get_tools()
tools_by_name = {t.name: t for t in tools}
print("Tools loaded via get_tools() (tools primitive only):")
for t in tools:
    print(f"  - {t.name}: {t.description.splitlines()[0]}")

Tools loaded via get_tools() (tools primitive only):
  - data_summary: Patients and features per omics view, plus PAM50 subtype class counts.
  - split_summary: Train/test patient counts and features kept per view after selection.
  - active_factors: Which MOFA factors are active and each factor's total R2 across views.
  - factor_view_r2: Variance explained (R2) by one factor in each omics view, and its top view.
  - factor_subtype_association: Rank active factors by eta-squared association with PAM50 subtype (train).
  - top_features_for_factor: Top positive/negative weighted features of a view for a factor (its drivers).
  - classify_subtype_from_factors: Predict subtype from MOFA factors; held-out metrics + most-confused pair.
  - train_vs_test_subtype_association: Compare factor<->subtype eta-squared on train vs projected test patients.


In [4]:
# Proof the adapter did NOT discard the other primitives — reachable via the same client.
res = await mcp_client.get_resources("mofa", uris=["mofa://summary"])
print("get_resources('mofa', ['mofa://summary']) ->")
print("  ", res[0].data[:80].replace(chr(10), " "), "...\n")

pm = await mcp_client.get_prompt("mofa", "interpret_factor", arguments={"factor": "Factor2"})
print("get_prompt('mofa', 'interpret_factor', {factor: Factor2}) ->")
print("  ", pm[0].content[:100])

get_resources('mofa', ['mofa://summary']) ->
   {   "n_patients": 603,   "views": [     "transcriptomics",     "proteomics",     ...



get_prompt('mofa', 'interpret_factor', {factor: Factor2}) ->
   Interpret MOFA Factor2. Report which omics views it explains (factor_view_r2), how strongly it assoc


## 4. Sanity-check one tool (no LLM)

Call a tool straight through the adapter so the structured evidence is visible
before involving the model.

In [5]:
def parse_mcp(result):
    """MCP tools return text content blocks; json.loads each back to a dict."""
    if isinstance(result, list):
        out = []
        for block in result:
            text = block.get("text") if isinstance(block, dict) else block
            out.append(json.loads(text) if isinstance(text, str) else text)
        return out
    return json.loads(result) if isinstance(result, str) else result

assoc = parse_mcp(await tools_by_name["factor_subtype_association"].ainvoke({}))
print("factor_subtype_association (top 3):", assoc[:3])

cls = parse_mcp(await tools_by_name["classify_subtype_from_factors"].ainvoke({}))
print("classify_subtype_from_factors    :", cls)

factor_subtype_association (top 3): [{'factor': 'Factor2', 'eta_squared': 0.743}, {'factor': 'Factor1', 'eta_squared': 0.35}, {'factor': 'Factor7', 'eta_squared': 0.264}]


classify_subtype_from_factors    : [{'metrics': {'accuracy': 0.828, 'balanced_accuracy': 0.741, 'macro_f1': 0.747}, 'most_confused': {'true': 'LumA', 'predicted': 'LumB', 'count': 9}}]


## 5. Bind the MCP tools to Claude

Identical to notebook 01 — `bind_tools` doesn't care the tools came from an MCP
server. Binding does not call the API.

In [6]:
from langchain_anthropic import ChatAnthropic

MODEL = "claude-haiku-4-5"   # swap for any current Claude model id
llm = ChatAnthropic(model=MODEL, temperature=0)
llm_with_tools = llm.bind_tools(tools)
print("Bound", len(tools), "MCP tools to", MODEL)

Bound 8 MCP tools to claude-haiku-4-5


## 6. Run the agent loop (async)  ⟵ *these cells spend API tokens*

Same loop as notebook 01, but `await`-ing the model and tools because MCP tools
are asynchronous.

In [7]:
from langchain_core.messages import AIMessage, HumanMessage, SystemMessage, ToolMessage

SYSTEM = ("You are a computational-biology assistant analysing a fitted MOFA model of "
          "TCGA breast-cancer multi-omics data. Use the tools to gather evidence; ground "
          "every quantitative claim in tool results and name which tool you used. Factors "
          "are 'Factor1'..'Factor10'; subtypes are PAM50 (LumA, LumB, Basal, Her2, Normal).")

async def run_agent(question: str, max_steps: int = 6, verbose: bool = True) -> AIMessage:
    messages = [SystemMessage(content=SYSTEM), HumanMessage(content=question)]
    for step in range(max_steps):
        ai = await llm_with_tools.ainvoke(messages)
        messages.append(ai)
        if not ai.tool_calls:
            return ai
        for call in ai.tool_calls:
            if verbose:
                print(f"[step {step}] -> {call['name']}({call['args']})")
            result = await tools_by_name[call["name"]].ainvoke(call["args"])
            messages.append(ToolMessage(content=json.dumps(result, default=str),
                                        tool_call_id=call["id"]))
    raise RuntimeError(f"Agent did not finish within {max_steps} steps.")

In [8]:
final = await run_agent(
    "Which MOFA factor is most associated with breast-cancer subtype, and which "
    "transcriptomic features most strongly drive it?")
print("\n=== FINAL ANSWER ===\n")
print(final.content)

[step 0] -> factor_subtype_association({})


[step 1] -> top_features_for_factor({'factor': 'Factor2', 'view': 'transcriptomics', 'n': 10})



=== FINAL ANSWER ===

## Summary

**Factor2 is the MOFA factor most strongly associated with breast-cancer subtype**, with an eta-squared of **0.743** (training data), far exceeding all other factors. This was determined using the `factor_subtype_association` tool, which ranked all active factors by their association strength with PAM50 subtype.

### Top Transcriptomic Drivers of Factor2

The `top_features_for_factor` tool identified the following genes as the strongest drivers:

**Top 5 Positive Contributors** (highest weights):
1. **ENSG00000160182.3** (weight: 0.470)
2. **ENSG00000173467.9** (weight: 0.469)
3. **ENSG00000082175.15** (weight: 0.436)
4. **ENSG00000235687.9** (weight: 0.424)
5. **ENSG00000160180.15** (weight: 0.417)

**Top 5 Negative Contributors** (lowest weights):
1. **ENSG00000166535.20** (weight: -0.337)
2. **ENSG00000186832.9** (weight: -0.301)
3. **ENSG00000102243.13** (weight: -0.283)
4. **ENSG00000185686.18** (weight: -0.279)
5. **ENSG00000135069.14** (weight:

## 7. Same answer as the direct-tools track

The flagship answer should match [`01_tools_v1`](01_tools_v1.ipynb) — same
backend and cached model, different transport. Compare the factor the agent
picked against the direct evidence:

In [9]:
print("Direct evidence (top subtype-associated factors):")
for row in assoc[:3]:
    print(f"  {row['factor']}: eta_squared = {row['eta_squared']}")

Direct evidence (top subtype-associated factors):
  Factor2: eta_squared = 0.743
  Factor1: eta_squared = 0.35
  Factor7: eta_squared = 0.264


## Reflection

- **Primitives:** name the three this server exposes. Which adapter calls load
  the resource and the prompt instead of the tools? (`get_resources()`, `get_prompt()`)
- **Did the adapter throw MCP away?** No — it *is* an MCP client speaking the
  section-2 protocol. What actually differs between sections 2 and 3? (convenience
  and object model, not capability.)
- **Layering:** MCP rode on `stdio` + JSON-RPC. What changes if the transport is
  HTTP? (the primitives don't.)
- **Annotations:** every tool was `readOnlyHint=True`. How would a client use
  that? What changes for a tool that *writes*?
- **Same answer:** it matched notebook 01. So what did MCP buy here, and when does
  it pay off? (big/private data behind an audited server, reuse across clients.)